In [2]:
import logging
import sys

# Konfiguration für Jupyter erzwingen
logging.basicConfig(
    level=logging.INFO,  # Auf welchen Level wird geloggt 
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout, 
    force=True         
)

# Logger erstellen
logger = logging.getLogger(__name__)

# Testen
logger.debug("Das ist eine Debug-Nachricht (nur für Entwickler).")
logger.info("Das Programm läuft ganz normal weiter.")
logger.warning("Achtung, hier könnte bald ein Problem auftreten!")

11:09:24 - __main__ - INFO - Das Programm läuft ganz normal weiter.
11:09:25 - __main__ - WARNING - Achtung, hier könnte bald ein Problem auftreten!


In [7]:
import pickle
import gc
import pandas as pd

# ---------------------------------------------------------
# Schritt 1: Master-Tabelle aufbauen (z.B. aus Population)
# ---------------------------------------------------------
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_population.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# Wir nehmen diesen DataFrame als unsere Basis (hier sind u, v, arc_id schon drin)
df_master_edges = temp_data['edges'][['arc_id', 'u', 'v', 'length_m', 'population']].copy()
df_master_nodes = temp_data['nodes'][['node', 'lat', 'lon']].copy()

# WICHTIG: Die große Original-Datei sofort wieder aus dem RAM werfen!
del temp_data
gc.collect()


# ---------------------------------------------------------
# Schritt 2: NUR die Unfalldaten (Accidents) extrahieren
# ---------------------------------------------------------
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_accidents.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# Wir extrahieren NUR die arc_id und den weighted_score
df_accidents_slim = temp_data['edges'][['arc_id', 'weighted_score']]

# Wir mergen den Score in unsere Master-Tabelle (anhand der arc_id)
df_master_edges = pd.merge(df_master_edges, df_accidents_slim, on='arc_id', how='left')

# Original-Datei wieder löschen
del temp_data, df_accidents_slim
gc.collect()


# ---------------------------------------------------------
# Schritt 3: NUR die Naturdaten (Nature) extrahieren
# ---------------------------------------------------------
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_with_nature.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# Wir extrahieren NUR die arc_id und den nature_score
df_nature_slim = temp_data['edges'][['arc_id', 'nature_score']]

# An die Master-Tabelle anfügen
df_master_edges = pd.merge(df_master_edges, df_nature_slim, on='arc_id', how='left')

# Original-Datei löschen
del temp_data, df_nature_slim
gc.collect()

with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/processed/all_data.pkl", 'rb') as f:
    all_data = pickle.load(f)

# ---------------------------------------------------------
# Schritt 4: Tunnel- und Straßendaten (Geo) extrahieren
# ---------------------------------------------------------
# Pfad ggf. anpassen auf deine germany_graph_geo_tun (1).pkl Datei
with open(r"/home/luca/Studium/4.Semester/Operations_Research/Routenplanung_Gefahrguttransport/data/germany_data/germany_graph_geo_tun_small.pkl", 'rb') as f:
    temp_data = pickle.load(f)

# 1. Tunnel-Dictionary in DataFrame umwandeln
# temp_data['tunnel'] sieht so aus: {0: 'no', 1: 'no', ...}
# .items() macht daraus Paare, die wir Spalten zuweisen können: 'arc_id' und 'tunnel'
df_tunnel_slim = pd.DataFrame(list(temp_data['tunnel'].items()), columns=['arc_id', 'tunnel'])

# 2. Highway-Dictionary in DataFrame umwandeln (Bonus für spätere ADR-Regeln!)
df_highway_slim = pd.DataFrame(list(temp_data['highway'].items()), columns=['arc_id', 'highway'])

# 3. Beide Tabellen an den Master-Graphen mergen (wie immer über die 'arc_id')
df_master_edges = pd.merge(df_master_edges, df_tunnel_slim, on='arc_id', how='left')
df_master_edges = pd.merge(df_master_edges, df_highway_slim, on='arc_id', how='left')

# 4. Speicher wieder sauber machen!
del temp_data, df_tunnel_slim, df_highway_slim
gc.collect()

# Zentraler Master-Container für alle Graphdaten
master_node = {
    "nodes": df_master_nodes,
    "edges": df_master_edges
}

# Legacy-Aliase für nachfolgende Zellen
df_nodes = master_node["nodes"]
df_edges = master_node["edges"]

/tmp/ipykernel_5458/617786986.py:24: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  temp_data = pickle.load(f)


In [8]:
import pandas as pd

print("="*90)
print("1. MASTER-GRAPH (KANTEN / EDGES)")
print("="*90)

# Welche Spalten haben wir jetzt alle erfolgreich zusammengeführt?
print(f"Spalten im Master-Graph:\n{df_master_edges.columns.tolist()}\n")

# Wie viele Kanten gibt es insgesamt?
print(f"Anzahl der Kanten im Netzwerk: {len(df_master_edges)}")

# Ein Blick auf die ersten 5 Zeilen (Alle wichtigen Metriken auf einen Blick!)
print(f"\nErste 5 Kanten (Head):\n{df_master_edges.head()}")

print("-" * 90)
print("STATISTIKEN FÜR DIE ZIELFUNKTION (Risk & Costs)")
print("-" * 90)

# Check der maximalen Scores (Wichtig für die spätere Normalisierung!)
print(f"Maximaler Unfall-Score (weighted_score): {df_master_edges['weighted_score'].max()}")
print(f"Maximaler Natur-Score (nature_score):    {df_master_edges['nature_score'].max()}")
print(f"Maximale Bevölkerung (population):       {df_master_edges['population'].max()}")
print(f"Längste Kante (length_m):              {df_master_edges['length_m'].max()} Meter")
# WICHTIG FÜR DEN SOLVER: Gibt es fehlende Werte (NaN)? 
# PuLP stürzt ab, wenn es mit NaN-Werten rechnen muss!
print("\nFehlende Werte (NaN) in den kritischen Spalten:")
print(df_master_edges[['weighted_score', 'nature_score', 'population', 'length_m']].isnull().sum())

# Falls es fehlende Werte gibt, füllen wir sie am besten mit 0 (oder einem anderen Standardwert)
# df_master_edges.fillna(0, inplace=True) 

print("="*90)
print("2. KNOTEN (NODES) & GEODATEN (aus master_node)")
print("="*90)
# Je nachdem, ob du die Knoten als DataFrame oder Dictionary hast:
# Falls Knoten im Master-Graph sind:
# print(f"Erste 5 Knoten:\n{df_master_nodes.head()}") 

# Tunnel-Status prüfen (Wichtig für ADR-Sperren)
# Da tunnel ein Dictionary war, schauen wir uns die Werte an:
if 'tunnel' in df_master_edges.columns:
    print(f"\nVerteilung der Tunnel im Netzwerk:\n{df_master_edges['tunnel'].value_counts()}")
else:
    print("\nHinweis: Tunnel-Daten sind noch nicht im df_master_edges (müssen ggf. noch gemergt werden).")

print("="*90)
print("3. KUNDEN, FAHRZEUGE & LIEFERUNGEN (ALL_DATA)")
print("="*90)

print(f"Verfügbare Datensätze in all_data: {list(all_data.keys())}\n")

print("Kundenstandorte (Erste 3):")
print(all_data['customer_locations'].head(3))
print(f"\nBeispiel-Ort (Index 1): {all_data['customer_locations'].iloc[1]['Ort']}\n")

print("-" * 40)
print("Fahrzeuge (Erste 3):")
print(all_data['vehicles'].head(3))

print("-" * 40)
print("Lieferungen (Erste 3):")
print(all_data['delivery_routes'].head(3))

1. MASTER-GRAPH (KANTEN / EDGES)
Spalten im Master-Graph:
['arc_id', 'u', 'v', 'length_m', 'population', 'weighted_score', 'nature_score', 'tunnel', 'highway']

Anzahl der Kanten im Netzwerk: 3025675

Erste 5 Kanten (Head):
   arc_id           u           v    length_m  population  weighted_score  \
0       0  3786023396  3786023162  111.365417         0.0             0.0   
1       1    10210552   277783046   41.566623        29.0             0.0   
2       2   277783046    10210552   41.566623        29.0             0.0   
3       3   277783046  5186817684   24.481156        29.0             0.0   
4       4  5186817684   277783046   24.481156        29.0             0.0   

   nature_score tunnel   highway  
0      0.000655     no  motorway  
1      0.001184     no   primary  
2      0.001184     no   primary  
3      0.001218     no   primary  
4      0.001218     no   primary  
------------------------------------------------------------------------------------------
STATISTIKEN 

In [ ]:
# ================================
#  TESTDATEN (NUR BERLIN)
# ================================

import pandas as pd

# Depot (Berlin Mitte)
depot_lat = 52.5200
depot_lon = 13.4050

# 3 Kunden in Berlin
df_customers = pd.DataFrame([
    {"Nr": 1, "Ort": "Berlin Kreuzberg", "Breitengrad": 52.4986, "Längengrad": 13.4030},
    {"Nr": 2, "Ort": "Berlin Charlottenburg", "Breitengrad": 52.5163, "Längengrad": 13.3041},
    {"Nr": 3, "Ort": "Berlin Pankow", "Breitengrad": 52.5695, "Längengrad": 13.4010},
])

# delivery_routes ersetzen
delivery_routes = pd.DataFrame([
    {"customer_id": 1, "danger_class": "3", "quantity": 10000},
    {"customer_id": 2, "danger_class": "2", "quantity": 8000},
    {"customer_id": 3, "danger_class": "8", "quantity": 5000},
])

# vehicles (minimal)
vehicles = all_data['vehicles']

# künstliches all_data
all_data = {
    "customer_locations": df_customers,
    "delivery_routes": delivery_routes,
    "vehicles": vehicles
}

In [9]:
# Berechnen des nächstgelegenen Knotens für die Lieferungen

import pandas as pd
from math import radians, sin, cos, sqrt, atan2


def haversine(lat1, lon1, lat2, lon2):
    """
    Berechnet Distanz zwischen zwei GPS-Punkten in Metern
    Haversine-Distanzfunktion
    """

    R = 6371000  # Erdradius in Metern

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2)
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


# Knotendaten aus master_node laden
df_nodes = master_node["nodes"]
df_edges = master_node["edges"]

#------------Depot auf nächsten Knoten mappen----------------

depot_node_result = None
min_distance = float("inf")

for _, node in df_nodes.iterrows():
    dist = haversine(depot_lat, depot_lon, node["lat"], node["lon"] )

    if dist < min_distance:
        min_distance = dist
        depot_node_result = node["node"]

print(f"Depot gemappt auf Knoten: {depot_node_result} ({round(min_distance,2)} m)")

# Kunden auf nächsten Knoten mappen
customer_to_node = {}
df_customers = all_data['customer_locations']  # Kundendaten

for _, customer in df_customers.iterrows():

    customer_lat = customer["Breitengrad"]
    customer_lon = customer["Längengrad"]

    min_distance = float("inf")
    nearest_node = None

    for _, node in df_nodes.iterrows():

        dist = haversine(
            customer_lat,
            customer_lon,
            node["lat"],
            node["lon"]
        )

        if dist < min_distance:
            min_distance = dist
            nearest_node = node["node"]

    customer_to_node[customer["Nr"]] = {
        "customer_name": customer["Ort"],
        "nearest_node": int(nearest_node),
        "distance_m": round(min_distance, 2)
    }

# ================================
# DEBUG OUTPUT
# ================================

print("\n=== Kunden-Mapping ===")
for cid, info in customer_to_node.items():
    print(f"Kunde {cid}: {info['customer_name']}")
    print(f"-> Knoten: {info['nearest_node']}")
    print(f"-> Distanz: {info['distance_m']} m")
    print("-" * 40)

# ================================
# DEPOT / ZIEL DEFINITION
# ================================

O_dep = {
    row["Nr"]: int(depot_node_result)
    for _, row in df_customers.iterrows()
}

D_dep = {
    cid: info["nearest_node"]
    for cid, info in customer_to_node.items()
}

NameError: name 'depot_lat' is not defined

In [ ]:
# ALLOW DICTIONARY
# Daten aus dem zentralen Master-Container verwenden

df_edges = master_node['edges']

# Für Kompatibilität mit der bisherigen Logik:
tunnel_data_raw = df_edges['tunnel'].fillna('no').to_dict()
arcs_data = df_edges[['u', 'v']].to_dict('records')

print(f"Kanten: {len(arcs_data):,}")
print(f"Tunnel-Einträge: {len(tunnel_data_raw):,}")

# 1. Tunnel-Mapping erzeugen
tunnel_category_dict = {}

for edge_idx, arc in enumerate(arcs_data):

    u = arc['u']
    v = arc['v']

    e = (u, v)

    tunnel_cat = tunnel_data_raw.get(edge_idx, 'no')

    tunnel_category_dict[e] = tunnel_cat

logger.debug(f"{len(tunnel_category_dict):,} Tunnel-Mappings erstellt")

# OSM Tunneltypen -> ADR Kategorien
tunnel_category_mapping = {

    # Kein Tunnel
    'no': 'A',
    '': 'A',
    None: 'A',

    # Leicht restriktiv
    'building_passage': 'B',

    # Moderat restriktiv
    'covered': 'C',

    # Starke Restriktionen
    'yes': 'D',
    'avalanche_protector': 'D',
}

# ADR Restriktionen
ADR_tunnel_restrictions = {

    'A': {k: True for k in K },

    'B': {k: True for k in K},

    'C': {
        '1.1D': False,
        '1.5D': False,
        '2': True,
        '2 (TOC)': True,
        '3': True,
        '4': True,
        '5': True,
        '6': True,
        '8': True,
        '9': True,
    },

    'D': {
        '1.1D': False,
        '1.5D': False,
        '2': True,
        '2 (TOC)': True,
        '3': True,
        '4': True,
        '5': True,
        '6': False,
        '8': True,
        '9': False,
    }
}

# Allow Dictionary erzeugen
Allow_calc = {}

for e in E:

    raw_tunnel_cat = tunnel_category_dict.get(e, 'no')

    adr_cat = tunnel_category_mapping.get(raw_tunnel_cat, 'A')

    for k in K:

        allowed = ADR_tunnel_restrictions[adr_cat].get(
            k,
            True
        )

        Allow_calc[(e, k)] = 1 if allowed else 0

logger.debug("Allow erstellt")
logger.debug(f"Einträge: {len(Allow_calc):,}")

# Beispiele
sample = list(Allow_calc.items())[:10]

for key, value in sample:
    logger.debug(f"{key}: {value}")

unique_tunnel = set(df_edges['tunnel'].dropna().tolist())
print(unique_tunnel)

print(f"df_nodes head: \n{df_nodes.head()}")

In [ ]:
import pulp as pl

logger.setLevel(logging.DEBUG)
# ----- SETS (MENGEN) & PARAMETER --------#

# --- SETS (Mengen) ---
V = [row['type'] for _, row in all_data['vehicles'].iterrows()] 
logger.debug(f"Fahrzeugtypen: {V}")
# Menge der verfügbaren Fahrzeuge  => ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']

L = [row['customer_id'] for _, row in all_data['delivery_routes'].iterrows()]
logger.debug(f"Lieferungen: {L}")
# [1, 2, 3, 4, 5]
# L =  all_data['delivery_routes']             
# Lieferungen (Standort(lat und long), un-Nummer,substanzname, Klasse, quantity, unit, packaging_type )

K = [all_data['delivery_routes'].iloc[i]['danger_class'] for i in range(len(all_data['delivery_routes']))] 
logger.debug(f"Gefahrgutklassen: {K}")        
# Gefahrgutklassen der einzelnen Lieferungen =>['3', '2 (TOC)', '3', '1.1D', '3']

# Richtiges Format für den Solver: [172539, 172541, 172542, ]
N = df_nodes['node'].tolist()
logger.debug(f"Anzahl Knoten im Straßennetz: {len(N)}\n N Header: {N[:5]}")

# Alle Knoten des Straßennetzes

# Richtiges Format für den Solver: [(1822620447, 11405216193), (11405216193, 2845478638), ]
E = list(zip(df_edges['u'], df_edges['v']))
logger.debug(f"Anzahl Kanten im Straßennetz: {len(E)}\n E Header: {E[:5]}")
# Gerichtete Kanten im Straßennetz 


# --- PARAMETER (Eingabedaten) ---

# Lieferungsdaten (Bezug zu L)
Dem = {row['customer_id']: row['quantity'] for _, row in all_data['delivery_routes'].iterrows()} 
logger.debug(f"Nachfrage je Lieferung: {Dem}")
# Gewicht je Lieferung (kg) => Annahme das 1 L = 1 Kg ist 
#{1: 15000, 2: 5000, 3: 21500, 4: 1500, 5: 1000, 6: 19500, 7: 15000, 8: 10000, 9: 2500, 10: 8000}}

Class = {row['customer_id']: row['danger_class'] for _, row in all_data['delivery_routes'].iterrows()} 
logger.debug(f"Gefahrgutklassen je Lieferung: {Class}")
# Gefahrgutklasse je Lieferung
#{1: '3', 2: '2 (TOC)', 3: '3', 4: '1.1D', 5: '3', 6: '8', 7: '9', 8: '2', 9: '3', 10: '9'}


depot_node = depot_node_result
logger.debug(f"Depot-Knoten: {depot_node}")
lieferungen = all_data['customer_locations']["Nr"]
# O_dep = {
#     lieferung_id: int(depot_node)
#     for lieferung_id in lieferungen
# } 
O_dep = O_dep
logger.debug(f"Startknoten (Origin) je Lieferung: {O_dep}")

# D_dep = {
#     customer_id: info["nearest_node"]
#     for customer_id, info in customer_to_node.items()
# }     # Zielknoten (Destination)    

D_dep = D_dep
logger.debug(f"Zielknoten (Destination) je Lieferung: {D_dep}")


ChargeNode = {n: 0 for n in N}
# Erstmal hat kein Knoten eine Ladestation, erst für Datensatz Deutschland werden sie hinzugefügt 


# Fahrzeugdaten (Bezug zu V) aus (vehicles.csv)
Cap = {row['type']: row['capacity_kg'] for _, row in all_data['vehicles'].iterrows()}   
logger.debug(f"Kapazität je Fahrzeugtyp: {Cap}") 
# Maximale Nutzlast (t) => {'MAN_eTGX': 18, 'Volvo_FH_Electric': 20, 'Mercedes_eActros_600': 22}  

Range = {row['type']: row['range_km'] for _, row in all_data['vehicles'].iterrows()}    
logger.debug(f"Akkureichweite je Fahrzeugtyp: {Range}")
# Akkureichweite => {'MAN_eTGX': 500, 'Volvo_FH_Electric': 470, 'Mercedes_eActros_600': 500} 

FC = {row['type']: row['fixed_cost'] for _, row in all_data['vehicles'].iterrows()}   
logger.debug(f"Fixkosten je Fahrzeugtyp: {FC}")
# Fixkosten (€/Tag) => {'MAN_eTGX': 180, 'Volvo_FH_Electric': 200, 'Mercedes_eActros_600': 220}

# --- Netzwerk- & Risikodaten (Bezug zu E und K) ---

Dist = {
    (row["u"], row["v"]): float(row["length_m"]) / 1000
    for _, row in df_edges.iterrows()
}
logger.debug(f"Distanz-Dictionary vor Kontraktion: {len(Dist):,} Einträge\n Dist Header: {list(Dist.items())[:5]}")
# Länge der Kante in km              

#Risiko muss hier berechnet werden mit parametern alpha, beta und gamma
Risk = Risk_calc          # Risikoscore [0, 1] je Kante und Klasse      
logger.debug(f"Risk Dictionary: {len(Risk):,} Einträge\n Risk Header: {list(Risk.items())[:5]}")
Allow = Allow_calc            # 1 wenn erlaubt, 0 wenn gesperrt (ADR)       
logger.debug(f"Allow Dictionary: {len(Allow):,} Einträge\n Allow Header: {list(Allow.items())[:5]}")
# Beispiel für eine Sperre: Allow[(('n1', 'ziel_A'), 'klasse_6')] = 0







# -------------- Kontraktion der Kanten und Knoten -----------
# Das heißt Knoten und Kanten, die an denen keine Entscheidung getroffen werden muss, da diese z.B. auf einer Straße ohne Abzweigungen liegen werden zusammen gefasst

import networkx as nx
# 1. Basis-Graphen aus den rohen Kanten und Distanzen bauen
G_raw = nx.DiGraph()
for (u, v), d in Dist.items():
    G_raw.add_edge(u, v, weight=d)

logger.debug(f"Roher Graph: {G_raw.number_of_nodes()} Knoten, {G_raw.number_of_edges()} Kanten")

# 2. Größte zusammenhängende Komponente finden (Die Insel-Lösung!)
# Stellt sicher, dass das Depot und alle Ziele auf einem verbundenen Netz liegen
components = list(nx.strongly_connected_components(G_raw))
largest_cc = max(components, key=len)

G_main = G_raw.subgraph(largest_cc).copy()
logger.debug(f"Hauptnetzwerk: {G_main.number_of_nodes()} Knoten, {G_main.number_of_edges()} Kanten")


# 1. Wir kopieren den Hauptgraphen, um sicher zu arbeiten
G_contract = G_main.copy()

# 2. Beschützte Knoten definieren (Depot + alle Zielknoten)
protected_nodes = set(O_dep.values()).union(set(D_dep.values()))

logger.debug(f"Knoten vor Kontraktion: {len(G_contract.nodes())}")
logger.debug(f"Kanten vor Kontraktion: {len(G_contract.edges())}")

# Kopieren von Risk und Allow, damit wir diese auch an die verknüpften Kanten anpassen können
Risk = Risk_calc.copy() if isinstance(Risk_calc, dict) else dict(Risk_calc)
Allow = Allow_calc.copy() if isinstance(Allow_calc, dict) else dict(Allow_calc)

# 3. Kontraktions-Schleife
contracted_count = 0
nodes_to_check = list(G_contract.nodes())

for node in nodes_to_check:
    # Überspringe Kunden und das Depot
    if node in protected_nodes:
        continue
        
    
    # Identifizierung ob ein Knoten ein Durchgangsknoten ist => 1 In und 1 Out 
    if G_contract.in_degree(node) == 1 and G_contract.out_degree(node) == 1:
        
        # Vorgänger (u) und Nachfolger (v)
        u = list(G_contract.predecessors(node))[0]
        v = list(G_contract.successors(node))[0]
        

        if u != v:
            # Distanzen der beiden Teilstücke abrufen
            dist_in = G_contract[u][node].get('weight', 0)
            dist_out = G_contract[node][v].get('weight', 0)
            new_dist = dist_in + dist_out
            
            new_risk = {}
            new_allow = {}
            for k in K:
                # Risiko aufsummieren
                r_in = Risk.get(((u, node), k), 0)
                r_out = Risk.get(((node, v), k), 0)
                new_risk[k] = r_in + r_out
                
                # Allow verknüpfen (Nur wenn BEIDE 1 sind, bleibt es 1)
                a_in = Allow.get(((u, node), k), 1)
                a_out = Allow.get(((node, v), k), 1)
                new_allow[k] = a_in * a_out
            
            # Kante aktualisieren oder neu anlegen
            if G_contract.has_edge(u, v):
                if new_dist < G_contract[u][v]['weight']:
                    G_contract[u][v]['weight'] = new_dist
                    for k in K:
                        Risk[((u, v), k)] = new_risk[k]
                        Allow[((u, v), k)] = new_allow[k]
            else:
                G_contract.add_edge(u, v, weight=new_dist)
                for k in K:
                    Risk[((u, v), k)] = new_risk[k]
                    Allow[((u, v), k)] = new_allow[k]
            # Den nutzlosen Zwischenknoten (und damit seine alten Kanten) löschen
            G_contract.remove_node(node)
            contracted_count += 1

logger.debug(f"Erfolgreich gelöschte Zwischenknoten: {contracted_count}")
logger.debug(f"Knoten nach Kontraktion: {len(G_contract.nodes())}")
logger.debug(f"Kanten nach Kontraktion: {len(G_contract.edges())}")


# 4. MILP-DATEN AKTUALISIEREN
# Jetzt überschreiben wir die Mengen für den Solver mit dem geschrumpften Graphen
E = list(G_contract.edges())
N = list(G_contract.nodes())
logger.debug(f"Finale Knoten (N): {len(N)}")
logger.debug(f"Finale Kanten (E): {len(E)}")
# Distanz-Dictionary neu aufbauen
Dist = {
    (u, v): data['weight']
    for u, v, data in G_contract.edges(data=True)
}
logger.debug(f"Dist Header: {list(Dist.items())[:5]}")







# Koordinaten-Dictionary aus deinen rohen Knotendaten erstellen
node_coords = df_nodes.set_index("node")[["lat", "lon"]].apply(tuple, axis=1).to_dict()

# Funktion zur Prüfung, ob eine Kante in der Ellipse/Box liegt
def in_bounding_box(u, v, start, ziel, buffer=0.2):
    # Fallback, falls Koordinaten fehlen (dann Kante zulassen)
    if u not in node_coords or v not in node_coords:
        return True 
        
    lat1, lon1 = node_coords[start]
    lat2, lon2 = node_coords[ziel]
    
    min_lat, max_lat = min(lat1, lat2), max(lat1, lat2)
    min_lon, max_lon = min(lon1, lon2), max(lon1, lon2)
    
    # Puffer berechnen (z.B. 20% der Distanz in alle Richtungen für Umwege)
    lat_buffer = max((max_lat - min_lat) * buffer, 0.02)
    lon_buffer = max((max_lon - min_lon) * buffer, 0.02)
    
    u_lat, u_lon = node_coords[u]
    v_lat, v_lon = node_coords[v]
    
    def check(lat, lon):
        return (min_lat - lat_buffer <= lat <= max_lat + lat_buffer) and \
               (min_lon - lon_buffer <= lon <= max_lon + lon_buffer)
               
    return check(u_lat, u_lon) and check(v_lat, v_lon)


# --- VARIABLEN ERSTELLEN (Nur Geografie-Filter, keine Sperren-Filter!) ---
valid_le_pairs = []
edges_per_l = {l: [] for l in L}
nodes_per_l = {l: set() for l in L}

logger.info("Generiere geografisch begrenzte Variablen-Matrix...")

for l in L:
    start = O_dep[l]
    ziel = D_dep[l]
    
    for (u, v) in E:
        # NUR Geografie prüfen. Erlaubt oder nicht entscheidet der Solver in C4!
        if in_bounding_box(u, v, start, ziel, buffer=0.3):
            valid_le_pairs.append((l, (u, v)))
            edges_per_l[l].append((u, v))
            nodes_per_l[l].update([u, v])

# Variable initialisieren
valid_ln_pairs = [(l, n) for l in L for n in nodes_per_l[l]]





# Fahrtkosten (Bezug zu V und E)
VC_per_km = all_data['vehicles'].set_index('type')['variable_cost_per_km'].to_dict()       #variable Kosten pro km je Fahrzeugtyp
Energy_per_km = all_data['vehicles'].set_index('type')['energy_kwh_per_km'].to_dict()      # Energieverbrauch pro km je Fahrzeugtyp (kWh/km)

strompreis = 0.35
# Strompreis erstmal fest lassen, weil wir nicht wissen ob Straßentyp an kanten dran steht 
VC = {}
for v in V:
    for e in E:
        # Dist[e] muss vorher definiert sein (z.B. als Dictionary Dist = {e: länge_in_km})
        distanz = Dist[e]
        
        # km-Kosten berechnen
        km_kosten = distanz * VC_per_km[v]
        
        # Energiekosten berechnen
        energie_kosten = distanz * Energy_per_km[v] * strompreis
        
        # Gesamte variable Kosten für dieses Fahrzeug auf dieser Kante
        VC[(v, e)] = km_kosten + energie_kosten

      # Variable Kosten (€) pro Fahrzeug und Kante

logger.debug(f"VC Dictionary: {len(VC):,} Einträge\n VC Header: {list(VC.items())[:5]}")

# Gewichtungsfaktoren für die Zielfunktion
w1 = 0.65  # Risiko-Gewicht
w2 = 0.35  # Kosten-Gewicht


logger.setLevel(logging.INFO)

In [ ]:
logger.setLevel(logging.DEBUG)
#----------------------  MODELL INITIALISIERUNG--------------------------#
model = pl.LpProblem("Gefahrgut_Routenplanung_MILP", pl.LpMinimize)



#---------------------- ENTSCHEIDUNGSVARIABLEN--------------------------#

# f[l, e]: Binär - 1, wenn Lieferung l über Kante e transportiert wird
#nutzen von valid_le_pairs aus dem Spatial Bounding, damit nur Variablen für geografisch relevante Kanten erzeugt werden
f = pl.LpVariable.dicts("Fluss", valid_le_pairs, cat='Binary')

# y[l, v]: Binär - 1, wenn Lieferung l dem Fahrzeug v zugewiesen wird
y = pl.LpVariable.dicts("Zuweisung", [(l, v) for l in L for v in V], cat='Binary')

# # z[v]: Binär - 1, wenn Fahrzeug v heute eingesetzt wird (Aktivierung)
# z = pl.LpVariable.dicts("Aktivierung", [v for v in V], cat='Binary')


# u[l, n]: Stetig - MTZ-Hilfsvariable für die Besuchsreihenfolge (Subtour-Eliminierung)
u = pl.LpVariable.dicts("MTZ", valid_ln_pairs, lowBound=0, upBound=len(N), cat='Continuous')

# q[l, n]: Stetig - Verbleibende Akkureichweite (State of Charge) in km am Knoten n
q = pl.LpVariable.dicts("SoC", valid_ln_pairs, lowBound=0, cat='Continuous')

# c[l, n]: Binär - 1, wenn Lieferung l am Knoten n nachgeladen wird
c = pl.LpVariable.dicts("Charge", valid_ln_pairs, cat='Binary')

# CostVar[l]: Stetig - Hilfsvariable zur Linearisierung der fahrzeugspezifischen Streckenkosten
CostVar = pl.LpVariable.dicts("CostVar", L, lowBound=0, cat='Continuous')

#----------------  ZIELFUNKTION (Normiert & Gewichtet)-------------------#

# Berechnung der theoretischen Maxima für Skalierung [0, 1]
Risk_max = max(Risk[(e, k)] for e in E for k in K)
Risk_sum_max = Risk_max * len(L) * len(E) 

Cost_max = (sum(FC[v] for v in V) 
          + sum(max(VC[(v, e)] for v in V) for e in E) * len(L))

# Einzelterme aufbauen
risk_term = pl.lpSum(Risk[(e, Class[l])] * f[(l, e)] for l in L for e in edges_per_l[l])

# fixed_cost_term = pl.lpSum(FC[v] * z[v] for v in V)
#Fixkosten pro Fahrt anstelle von pro Fahrzeug
fixed_cost_term = pl.lpSum(FC[v] * y[(l, v)] for l in L for v in V)

variable_cost_term = pl.lpSum(
    VC[(v, e)] * y[(l, v)] * (1 / len(V)) 
    for l in L for v in V for e in edges_per_l[l] 
    if Allow.get((e, Class[l]), 1) == 1
)

variable_cost_term = pl.lpSum(CostVar[l] for l in L)

# Dem Modell hinzufügen
model += (
    (w1 / Risk_sum_max) * risk_term + 
    (w2 / Cost_max) * (fixed_cost_term + variable_cost_term)
), "Minimiere_Risiko_und_Kosten_normiert"



#--------------- NEBENBEDINGUNGEN (Constraints) --------------------------#


# --- C1: Transportpflicht ---
# Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden.
for l in L:
    model += pl.lpSum(y[(l, v)] for v in V) == 1, f"C1_Transportpflicht_{l}"

# --- C2: Kapazitätsbedingung ---
# Die Summe der Liefermengen, die einem Fahrzeug zugewiesen werden, darf dessen Kapazität nicht überschreiten.
for l in L:
    for v in V:
        model += Dem[l] * y[(l, v)] <= Cap[v], f"C2_Kapazitaet_{l}_{v}"


# --- C4: Gefahrgutrestriktion ---
# Eine Kante darf von einer Lieferung nicht befahren werden, wenn sie gesperrt ist.
# (Es werden aus Performance-Gründen nur Constraints für gesperrte Kanten erzeugt).
for l in L:
    for e in edges_per_l[l]:
        if Allow.get((e, Class[l]), 1) == 0:
            model += f[(l, e)] <= 0, f"C4_Gefahrgut_Sperre_{l}_{e[0]}_{e[1]}"

# --- C5: Flusserhaltung (Network Flow Conservation) ---
# Garantiert eine zusammenhängende Route vom Start- zum Zielknoten.
for l in L:
    relevant_edges = edges_per_l[l]
    relevant_nodes = nodes_per_l[l]
    
    in_edges_l = {n: [] for n in relevant_nodes}
    out_edges_l = {n: [] for n in relevant_nodes}
    
    # HIER FIX: n1 und n2 statt u und v nutzen!
    for (n1, n2) in relevant_edges:
        in_edges_l[n2].append((n1, n2))
        out_edges_l[n1].append((n1, n2))

    flow_active = pl.lpSum(y[(l, v)] for v in V)

    for n in relevant_nodes:
        inflow = pl.lpSum(f[(l, e)] for e in in_edges_l[n])
        outflow = pl.lpSum(f[(l, e)] for e in out_edges_l[n])

        if n == O_dep[l]:      
            model += outflow - inflow == flow_active, f"C5_Flow_Start_{l}_{n}"
        elif n == D_dep[l]:    
            model += inflow - outflow == flow_active, f"C5_Flow_Ziel_{l}_{n}"
        else:                  
            model += inflow == outflow, f"C5_Flow_Trans_{l}_{n}"

# --- C6: Subtour-Eliminierung (Miller-Tucker-Zemlin) ---
# Verhindert, dass der Solver isolierte, physikalisch unvollständige Kreise (Schleifen) bildet.
for l in L:
    relevant_edges = edges_per_l[l]
    M_mtz = len(nodes_per_l[l])  
    
    for (i, j) in relevant_edges:
        if j != O_dep[l]:  
            model += u[(l, i)] - u[(l, j)] + M_mtz * f[(l, (i, j))] <= M_mtz - 1, f"C6_MTZ_{l}_{i}_{j}"

# --- C7: Ladezustand (SoC Tracking) ---
# Ausreichend großes Big-M für Reichweiten (z.B. das Doppelte der maximalen LKW-Reichweite)
M_soc = max(Range.values()) * 2  
for l in L:
    for (i, j) in edges_per_l[l]:
        model += q[(l, j)] <= q[(l, i)] - Dist[(i, j)] + M_soc * (1 - f[(l, (i, j))]) + M_soc * c[(l, i)], f"C7_SoC_{l}_{i}_{j}"


# --- C8: Akkukapazität ---
for l in L:
    for n in nodes_per_l[l]:
        model += q[(l, n)] <= pl.lpSum(Range[v] * y[(l, v)] for v in V), f"C8_Kapazitaet_{l}_{n}"

# --- C9: Ladeinfrastruktur ---
for l in L:
    for n in nodes_per_l[l]:
        model += c[(l, n)] <= ChargeNode.get(n, 0), f"C9_Laden_{l}_{n}"

# --- C10: Kosten-Linearisierung ---
# Großes M für Kosten, das alle Kanten abdeckt
M_cost = Cost_max 
for l in L:
    for v in V:
        route_cost = pl.lpSum(VC[(v, e)] * f[(l, e)] for e in edges_per_l[l])
        model += CostVar[l] >= route_cost - M_cost * (1 - y[(l, v)]), f"C10_CostLin_{l}_{v}"


In [ ]:
# --------------------- MODELL LÖSEN --------------------------------#
logger.debug("Ab jetzt startet Solver => 5 Minuten")
solver = pl.PULP_CBC_CMD(msg=1, timeLimit=300)  # 5 Minuten Zeitlimit
status = model.solve(solver)
print(f"Status: {pl.LpStatus[status]}")

logger.setLevel(logging.INFO)

In [ ]:
# ----------------ERGEBNIS-AUSWERTUNG --------------------------#

if pl.LpStatus[model.status] == 'Optimal':
    logger.info("OPTIMALE LOESUNG GEFUNDEN! Werte die Routen aus...\n")

    total_vc = 0
    total_risk = 0
    summary_table_data = []

    for l in L:
        print(f"\n LIEFERUNG {l} | Gefahrgutklasse: {Class[l]} | Gewicht: {Dem[l]} kg")
        print("-" * 60)
        
        # GEÄNDERT: Float-Toleranz > 0.5 für die y-Variable
        zugewiesenes_v = None
        for v in V:
            if pl.value(y[(l, v)]) is not None and pl.value(y[(l, v)]) > 0.5:
                zugewiesenes_v = v
                break
        
        if not zugewiesenes_v:
            print("   Fehler: Kein Fahrzeug für diese Lieferung zugewiesen!")
            continue
            
        print(f"   Fahrzeug: {zugewiesenes_v} (Reichweite: {Range[zugewiesenes_v]} km, Kapazitaet: {Cap[zugewiesenes_v]} kg)")

        # 2. Welche Kanten wurden befahren? (f-Variable)
        route_edges = []
        lieferung_dist = 0
        lieferung_risk = 0
        lieferung_vc = 0
        
        for e in edges_per_l[l]:
            if pl.value(f[(l, e)]) is not None and pl.value(f[(l, e)]) > 0.5:  
                route_edges.append(e)
                lieferung_dist += Dist[e]
                lieferung_risk += Risk.get((e, Class[l]), 0)
                lieferung_vc += VC[(zugewiesenes_v, e)]
        
        # 3. Kanten in die richtige Reihenfolge bringen (Start -> Ziel)
        geordnete_route = []
        aktueller_knoten = O_dep[l]
        ziel_knoten = D_dep[l]
        
        verfuegbare_kanten = route_edges.copy()
        
        for _ in range(len(route_edges)):
            naechste_kante = next((kante for kante in verfuegbare_kanten if kante[0] == aktueller_knoten), None)
            
            if naechste_kante:
                verfuegbare_kanten.remove(naechste_kante)
                geordnete_route.append(naechste_kante)
                aktueller_knoten = naechste_kante[1] 
                
                if aktueller_knoten == ziel_knoten:
                    break
                    
        # 4. Route ausgeben
        print(f"   Route ({len(geordnete_route)} Kanten):")
        if geordnete_route:
            knoten_liste = [str(geordnete_route[0][0])] + [str(kante[1]) for kante in geordnete_route]
            chunk_size = 8
            chunks = [" -> ".join(knoten_liste[i:i+chunk_size]) for i in range(0, len(knoten_liste), chunk_size)]
            route_string = " -> \n      ".join(chunks)
            print(f"      {route_string}")
        else:
            print("      Konnte die Route nicht chronologisch sortieren (Fallback-Ansicht):")
            fallback_string = " | ".join([f"{kante[0]} -> {kante[1]}" for kante in route_edges])
            print(f"      {fallback_string}")


        # 5. Metriken pro Lieferung ausgeben
        print(f"\n   Zusammenfassung fuer Lieferung {l}:")
        print(f"      - Gesamtdistanz:      {lieferung_dist:.2f} km")
        print(f"      - Variable Kosten:    {lieferung_vc:.2f} EUR")
        print(f"      - Kumuliertes Risiko: {lieferung_risk:.4f}")
        
        total_vc += lieferung_vc
        total_risk += lieferung_risk
        
        # Daten fuer die finale Tabelle speichern
        summary_table_data.append({
            'Lieferung': l,
            'Fahrzeug': zugewiesenes_v,
            'Fixkosten': FC[zugewiesenes_v],
            'VarKosten': lieferung_vc,
            'Risiko': lieferung_risk
        })
        
    # --- Gesamtsystem-Auswertung (GEÄNDERT) ---
   
    print("\nGESAMTSYSTEM")
    
    # Welche Fahrzeugtypen wurden insgesamt (evtl. mehrfach) genutzt?
    genutzte_typen = list(set([row['Fahrzeug'] for row in summary_table_data]))
    
    # Summiere die Fixkosten aus der Tabelle (jede Fahrt kostet!)
    total_fc = sum(row['Fixkosten'] for row in summary_table_data)
    
    print(f" Eingesetzte Fahrzeugtypen: {len(genutzte_typen)}")
    print(f" Namen:                     {', '.join(genutzte_typen)}")
    print(f" Gesamte Fixkosten:         {total_fc:.2f} EUR")
    print(f" Gesamte Var. Kosten:       {total_vc:.2f} EUR")
    print(f" TOTALE KOSTEN:             {total_fc + total_vc:.2f} EUR")
    print(f" TOTALES RISIKO:            {total_risk:.4f}")
    print("="*85 + "\n")

    # --- Zusammenfassende Tabelle ---
    print(" ZUSAMMENFASSUNG: ZUWEISUNG UND KOSTEN")
    print("-" * 85)
    print(f"| {'Lieferung':<10} | {'Fahrzeug':<25} | {'Fixkosten':<10} | {'Var. Kosten':<11} | {'Risiko':<8} |")
    print("-" * 85)
    for row in summary_table_data:
        print(f"| {str(row['Lieferung']):<10} | {row['Fahrzeug']:<25} | {row['Fixkosten']:>6.2f} EUR | {row['VarKosten']:>7.2f} EUR | {row['Risiko']:>8.4f} |")
    print("-" * 85)

else:
    logger.error(f"Keine verwertbare Route gefunden. Status ist: {pl.LpStatus[model.status]}")
    if pl.LpStatus[model.status] == 'Infeasible':
        print("Status: Infeasible - Das Modell hat keine Lösung gefunden, die alle Nebenbedingungen erfüllt")

In [ ]:
print("\n" + "="*80)
print("CONSTRAINT ANALYSE")
print("="*80)

for name, constraint in model.constraints.items():
  
    lhs_value = constraint.value()

    
    # Prüfe Verletzung
    violated = False

    # <=
    if constraint.sense == -1:
        # if lhs_value > 0:
        #     violated = True
        if lhs_value > 1e-4: 
            violated = True

    # >=
    elif constraint.sense == 1:
        if lhs_value < 0:
            violated = True
            

    # ==
    elif constraint.sense == 0:
        if abs(lhs_value) > 1e-6:
            violated = True

    if violated:
        print(f"\nConstraint: {name}")
        print(f"Value: {lhs_value}")
        print(f"Sense: {constraint.sense}")
        print(f"RHS: {-constraint.constant}")

        print(">>> VERLETZT <<<")